In [1]:
import torch

# 初始化权重参数
w = torch.tensor([1.0], requires_grad=True,dtype=torch.float32)
# 实例化优化器
optimizer = torch.optim.Adam([w],lr=0.01,betas=[0.9,0.999])
# 迭代的次数
count =3
for i in range(count):
    # 目标函数
    y = (w**2)/2
    # 清空梯度
    optimizer.zero_grad()
    # 计算梯度
    y.backward()
    # 更新参数
    optimizer.step()
    # 打印结果
    print(f'梯度:{w.grad.item():.3f}, 权重:{w.item():.3f}')

梯度:1.000, 权重:0.990
梯度:0.990, 权重:0.980
梯度:0.980, 权重:0.970


### **Adam 优化器更新规则**

Adam 优化器结合了动量法和 RMSProp，更新规则如下：

1. **计算梯度**：
   $$ g_t = \nabla_w y = w $$

2. **更新一阶动量（梯度的指数移动平均）**：
   $$ m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t $$
   其中，$\beta_1 = 0.9$，初始时 $ m_0 = 0 $。

3. **更新二阶动量（梯度平方的指数移动平均）**：
   $$ v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2 $$
   其中，$\beta_2 = 0.999$，初始时 $ v_0 = 0 $。

4. **偏差校正**（Adam 在早期迭代中对动量估计有偏差，需要校正）：
   $$ \hat{m}_t = \frac{m_t}{1 - \beta_1^t} $$
   $$ \hat{v}_t = \frac{v_t}{1 - \beta_2^t} $$
   其中 $ t $ 是迭代次数（从 1 开始）。

5. **更新参数**：
   $$ w_{t+1} = w_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} $$
   其中：
   - $\eta = 0.01$（学习率）
   - $\epsilon = 10^{-8}$（Adam 默认值，避免除零）
   - $\sqrt{\hat{v}_t}$ 是二阶动量的平方根。

---

### **三次迭代的计算过程**

以下是手动计算三次迭代的详细过程，设初始 $ w = 1.0 $，$\eta = 0.01$，$\beta_1 = 0.9$，$\beta_2 = 0.999$，$\epsilon = 10^{-8}$。

#### **第 1 次迭代**

- **初始值**：
  - $ w = 1.0 $
  - $ m_0 = 0 $
  - $ v_0 = 0 $
  - $ t = 1 $

- **目标函数和梯度**：
  $$ y = \frac{w^2}{2} = \frac{1.0^2}{2} = 0.5 $$
  $$ g_1 = w = 1.0 $$

- **更新一阶动量**：
  $$ m_1 = \beta_1 \cdot m_0 + (1 - \beta_1) \cdot g_1 = 0.9 \cdot 0 + (1 - 0.9) \cdot 1.0 = 0.1 $$

- **更新二阶动量**：
  $$ v_1 = \beta_2 \cdot v_0 + (1 - \beta_2) \cdot g_1^2 = 0.999 \cdot 0 + (1 - 0.999) \cdot 1.0^2 = 0.001 $$

- **偏差校正**：
  $$ \hat{m}_1 = \frac{m_1}{1 - \beta_1^1} = \frac{0.1}{1 - 0.9} = 1.0 $$
  $$ \hat{v}_1 = \frac{v_1}{1 - \beta_2^1} = \frac{0.001}{1 - 0.999} = 1.0 $$

- **更新权重**：
  $$ w_2 = w_1 - \eta \cdot \frac{\hat{m}_1}{\sqrt{\hat{v}_1} + \epsilon} = 1.0 - 0.01 \cdot \frac{1.0}{\sqrt{1.0} + 10^{-8}} \approx 1.0 - 0.01 \cdot \frac{1.0}{1.0} = 1.0 - 0.01 = 0.99 $$

- **输出**：
  - 梯度：$ w.grad = g_1 = 1.0 $
  - 权重：$ w = 0.99 $

#### **第 2 次迭代**

- **初始值**：

  - $ w = 0.99 $
  - $ m_1 = 0.1 $
  - $ v_1 = 0.001 $
  - $ t = 2 $

- **目标函数和梯度**：
  $$ y = \frac{w^2}{2} = \frac{0.99^2}{2} \approx 0.49005 $$
  $$ g_2 = w = 0.99 $$

- **更新一阶动量**：
  $$ m_2 = \beta_1 \cdot m_1 + (1 - \beta_1) \cdot g_2 = 0.9 \cdot 0.1 + (1 - 0.9) \cdot 0.99 = 0.09 + 0.099 = 0.189 $$

- **更新二阶动量**：
  $$ v_2 = \beta_2 \cdot v_1 + (1 - \beta_2) \cdot g_2^2 = 0.999 \cdot 0.001 + (1 - 0.999) \cdot 0.99^2 $$
  $$ = 0.000999 + 0.001 \cdot 0.9801 \approx 0.000999 + 0.0009801 = 0.0019791 $$

- **偏差校正**：
  $$ \hat{m}_2 = \frac{m_2}{1 - \beta_1^2} = \frac{0.189}{1 - 0.9^2} = \frac{0.189}{1 - 0.81} = \frac{0.189}{0.19} \approx 0.9947368 $$
  $$ \hat{v}_2 = \frac{v_2}{1 - \beta_2^2} = \frac{0.0019791}{1 - 0.999^2} = \frac{0.0019791}{1 - 0.998001} \approx \frac{0.0019791}{0.001999} \approx 0.990045 $$

- **更新权重**：
  $$ w_3 = w_2 - \eta \cdot \frac{\hat{m}_2}{\sqrt{\hat{v}_2} + \epsilon} = 0.99 - 0.01 \cdot \frac{0.9947368}{\sqrt{0.990045} + 10^{-8}} $$
  $$ \sqrt{0.990045} \approx 0.995013 $$

  $$ w_3 \approx 0.99 - 0.01 \cdot \frac{0.9947368}{0.995013} \approx 0.99 - 0.01 \cdot 0.999723 \approx 0.99 - 0.00999723 \approx 0.98000277 $$

- **输出**：

  - 梯度：$ w.grad = g_2 = 0.99 $
  - 权重：$ w \approx 0.980 $

#### **第 3 次迭代**

- **初始值**：

  - $ w \approx 0.98000277 $
  - $ m_2 = 0.189 $
  - $ v_2 \approx 0.0019791 $
  - $ t = 3 $

- **目标函数和梯度**：
  $$ y = \frac{w^2}{2} \approx \frac{0.98000277^2}{2} \approx 0.4801007 $$
  $$ g_3 \approx 0.98000277 $$

- **更新一阶动量**：
  $$ m_3 = \beta_1 \cdot m_2 + (1 - \beta_1) \cdot g_3 = 0.9 \cdot 0.189 + (1 - 0.9) \cdot 0.98000277 $$
  $$ = 0.1701 + 0.1 \cdot 0.98000277 \approx 0.1701 + 0.098000277 \approx 0.268100277 $$

- **更新二阶动量**：
  $$ v_3 = \beta_2 \cdot v_2 + (1 - \beta_2) \cdot g_3^2 = 0.999 \cdot 0.0019791 + (1 - 0.999) \cdot 0.98000277^2 $$
  $$ g_3^2 \approx 0.96040542 $$
  $$ v_3 \approx 0.999 \cdot 0.0019791 + 0.001 \cdot 0.96040542 \approx 0.0019771209 + 0.00096040542 \approx 0.0029375263 $$

- **偏差校正**：
  $$ \hat{m}_3 = \frac{m_3}{1 - \beta_1^3} = \frac{0.268100277}{1 - 0.9^3} = \frac{0.268100277}{1 - 0.729} = \frac{0.268100277}{0.271} \approx 0.989298 $$

  $$ \hat{v}_3 = \frac{v_3}{1 - \beta_2^3} = \frac{0.0029375263}{1 - 0.999^3} \approx \frac{0.0029375263}{1 - 0.997002999} \approx \frac{0.0029375263}{0.002997001} \approx 0.980005 $$

- **更新权重**：
  $$ w_4 = w_3 - \eta \cdot \frac{\hat{m}_3}{\sqrt{\hat{v}_3} + \epsilon} \approx 0.98000277 - 0.01 \cdot \frac{0.989298}{\sqrt{0.980005} + 10^{-8}} $$
  $$ \sqrt{0.980005} \approx 0.989951 $$

  $$ w_4 \approx 0.98000277 - 0.01 \cdot \frac{0.989298}{0.989951} \approx 0.98000277 - 0.01 \cdot 0.999341 \approx 0.98000277 - 0.00999341 \approx 0.97000936 $$

- **输出**：

  - 梯度：$ w.grad \approx 0.980 $
  - 权重：$ w \approx 0.970 $

---

### **代码运行输出**

运行代码后，输出如下（与计算结果一致，保留 3 位小数）：

```
梯度:1.000, 权重:0.990
梯度:0.990, 权重:0.980
梯度:0.980, 权重:0.970
```

应学习率加速了梯度下降，使得参数更新更稳定和高效。